# Dubai Real Estate 2025 - Developer Extraction
This notebook documents the creation of a new developer-level feature for Dubai’s 2025 real estate transactions, consolidating information from the heterogeneous text fields Project, Building, and Master Project wherever possible, providing an interpretable and economically meaningful variable for subsequent statistical and predictive analyses.

## Main Steps

**- Text Preprocessing:** All relevant text fields were lowercased, concatenated and normalized to ensure consistent tokenization.

**- Pattern Extraction:** Developer names were first detected through explicit textual patterns (e.g., “by …”, “developed by …”).

**- Fuzzy Matching:** Unmatched entries were aligned against a curated list of Dubai developers using similarity metrics to correct spelling and format variations.

**- Manual Validation:** The most frequent unmatched projects were manually mapped based on authoritative online sources.

**- Iterative Refinement:** Additional mappings were propagated via project and building name similarities, excluding ambiguous or misleading matches.

**- Final Integration:** Clean developer names were merged back into the master dataset, replacing null values while preserving the original project identifiers.

## Outcome

**Final developer coverage:** ≈ 69 % of all transactions (up from ~36 % in early automated extraction).  
≈ 68% of coverage for Units  
≈ 75% of coverage for Villas

**Model validation:** inclusion of the developer variable increased adjusted R² from 0.50 → 0.63, and feature importance in a random forest model showed ≈ 20 %. This was performed in a separated notebook for personal research porpouses and is therefore not included.

These results confirm that developer identity is a substantive market driver, capturing brand-, quality- and location-related effects not explained by property type, size or area alone.  
This final version provides a reliable, interpretable and quantitatively validated Developer feature suitable for both explanatory analysis and predictive modeling.

# Libraries

In [1]:
import pandas as pd # Data Manipulation
import numpy as np # Data Manipulation

import re
from rapidfuzz import fuzz
from itertools import combinations

from rapidfuzz import process, fuzz

# Data

In [3]:
raw = pd.read_csv('re_2025_analysis_project.csv')
df = raw.copy()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220918 entries, 0 to 220917
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_type  220918 non-null  object 
 1   procedure         220918 non-null  object 
 2   date              220918 non-null  object 
 3   property_type     220918 non-null  object 
 4   registration      220918 non-null  object 
 5   area              220918 non-null  object 
 6   building          194538 non-null  object 
 7   project           197958 non-null  object 
 8   master_project    194085 non-null  object 
 9   landmark          220918 non-null  object 
 10  metro             220918 non-null  object 
 11  mall              220918 non-null  object 
 12  rooms             220918 non-null  object 
 13  parking           220918 non-null  int64  
 14  size              220918 non-null  float64
 15  price             220918 non-null  float64
 16  meter_price       22

# 0. Initial Preprocessing

## 0.1. Normalize NaNs
Most missing values in the dataset have already been dealt with. The missing ones, left on purpose for this section, which are contained in the columns 'building', 'project' and 'master_project' are converted to empty strings to be able to perform the developer extraction techniques.

In [10]:
df = df.fillna('').astype(str)

## 0.2. Normalize Text
Apply basic text normalizing techniques to facilitate the text handling.

In [13]:
# --- Normalize all text ---
def normalize_text(s):
    """
    Clean and standardize strings for text matching.
    - Convert to lowercase
    - Remove excessive whitespace
    - Strip leading/trailing spaces
    """
    s = str(s).lower().strip()
    s = re.sub(r'\s+', ' ', s)       # collapse multiple spaces
    return s

In [15]:
for col in ['building', 'project', 'master_project']:
    df[col] = df[col].apply(normalize_text)

## 0.3. Creating Subset
Create a new DataFrame with the interesting variables that will be utilised during the 'developer' extraction process.

In [17]:
# Subset and copy (avoid modifying original)
df_dev = df[['building', 'project', 'master_project']].copy()

## 0.4. Concatanating Columns

In [21]:
# --- Combine all 3 columns into a single searchable text field ---
# Starting with building
df_dev['proj_concat'] = (
    df_dev['building'] + ' | ' +
    df_dev['project'] + ' | ' +
    df_dev['master_project']
).str.strip()

## 0.5. Deduplication

In [24]:
# --- Deduplication for faster exploration ---
projects_unique = df_dev['proj_concat'].drop_duplicates().reset_index(drop=True)

print(f"Unique project strings: {len(projects_unique)}")
projects_unique.tail(5)

Unique project strings: 4403


4398    ivy at park five | ivy at park five | internat...
4399             | park villa's | jumeirah village circle
4400                 plaza boutique - 4 |  | business bay
4401    urbana iii stacked house block-32 | urbana iii...
4402    park lane ? townhouses | park lane | dubai hil...
Name: proj_concat, dtype: object

# 1. Exact list matching
Create a list of known developer companies operating in Dubai. This list was obtained from online reliable sources.

The new 'proj_concat' string is compared against this list of developers, allowing to obtain an initial 29.1% of coverage.

In [27]:
developers = [
    "emaar", "damac", "nakheel", "sobha", "meraas", "azizi",
    "aldar", "dubai properties", "union properties", "nshama",
    "deyaar", "majid al futtaim", "mag", "ellington", "select group",
    "binghatti", "danube", "omniyat", "tiger properties", "wasl",
    "meydan", "al habtoor", "arenco", "arady", "rak properties",
    "reportage properties", "dubai holding", "al ghurair",
    "tecom group", "limitless", "seven tides", "dubai south",
    "time properties", "reef real estate", "fam properties",

    "imkan", "imtilak", "arista", "arada", "liwan", "khalifa bin dasmal",
    "skyline builders", "gemini property developers", "azco real estate",
    "lazourde", "oman properties", "samana", "gulf general investments",
    "al wasl properties", "khamas group", "empire development",
    "srk real estate", "daark real estate", "metac properties",
    "creative cluster authority", "palma holding", "dar al arkan",
    "centurion group", "premier developers", "symphony developers",
    "bloom properties", "eagle hills", "liv developers", "tabeer",
    "ramhan island development", "trident", "zaya developers",
    "merlin developers", "eagle properties", "al barari", 'irth',
    "tebyan", "cayan group", "omnia developments", "artar",
    "tanmiyat", "mirage", "decent real estate", "serenia developers",

    "avenew development", "ohana development", "pasha one",
    "valores property development", "richmind development",
    "confident group", "vision developments", "metac",
    "patriot developers", "aqua properties", "centurion",
    "sycamore developments", "ghrei development", "dar global",
    "avenue property", "reva developers", "q development",
    "arabtec", "emaar misr", "universal properties", "realty force",
    "signature developers", "the first group", "oriental real estate",
    "driven properties", "skyline developers", "jumeirah golf estates"
]

In [29]:
# Compile a regex that matches any developer word
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, developers)) + r')\b')

def extract_dev_exact(text):
    """
    Return the first developer name that matches exactly
    one of the known developer strings.
    """
    m = pattern.search(text)
    return m.group(1) if m else np.nan

# Apply on the deduplicated list first (much faster)
proj_df = pd.DataFrame({'proj_concat': projects_unique})
proj_df['dev_exact'] = proj_df['proj_concat'].apply(extract_dev_exact)

# Merge matches back to the full df_dev
df_dev = df_dev.merge(proj_df, on='proj_concat', how='left')

# coverage
print("Exact-match coverage:",
      f"{df_dev['dev_exact'].notna().mean()*100:.1f}% of records")

Exact-match coverage: 29.1% of records


# 2. Rule / Pattern Matching

In [32]:
# --- Combine all 3 columns into a single searchable text field ---
# Starting with master_project
df_dev['proj_concat'] = (
    df_dev['master_project'] + ' | ' +
    df_dev['project'] + ' | ' +
    df_dev['building']
).str.strip()

In [34]:
pattern_by_general = re.compile(r'\bby\s+([a-z0-9&\s\-\.]{2,60})', flags=re.IGNORECASE)

def extract_dev_pattern_general(text):
    m = pattern_by_general.search(text)
    if m:
        return m.group(1).strip()
    return np.nan

# Apply on the deduplicated list first (much faster)
projects_unique = df_dev['proj_concat'].drop_duplicates().reset_index(drop=True)
proj_df = pd.DataFrame({'proj_concat': projects_unique})

proj_df['dev_pattern2'] = proj_df['proj_concat'].apply(extract_dev_pattern_general)
df_dev = df_dev.merge(proj_df[['proj_concat','dev_pattern2']], on='proj_concat', how='left')

# Combine: prefer exact match, else pattern match
df_dev['dev_raw'] = df_dev['dev_exact'].fillna(df_dev['dev_pattern2'])

print("Pattern-based new coverage:",
      f"{df_dev['dev_raw'].notna().mean()*100:.1f}% of records")

Pattern-based new coverage: 36.3% of records


In [36]:
df_dev['dev_raw'].value_counts()

dev_raw
binghatti                              19156
damac                                  10141
sobha                                   8189
azizi                                   5863
danube                                  4971
                                       ...  
london gate real estate development        5
nexus                                      4
al marina                                  4
newbury developments                       1
casa vista development                     1
Name: count, Length: 122, dtype: int64

# 3. Manual Mapping

In [39]:
# Identify unmatched project strings (no developer found after fuzzy and pattern matching)
#unmatched = df_dev[df_dev['dev_raw'].isna()]

In [41]:
# Export top unmatched project strings for manual lookup
#unmatched = df_dev[df_dev['dev_raw'].isna()]
#top_unmatched = unmatched['proj_concat'].value_counts().reset_index().rename(columns={'index':'proj_concat','proj_concat':'count'})
#top_unmatched.to_csv('top_unmatched_projects.csv', index=False)
# print("Exported top_unmatched_projects.csv — inspect top rows.")
# top_unmatched.head(50)

In [43]:
manual_map = pd.read_csv('developer_manual_mapping.csv')
df_dev = df_dev.merge(manual_map, on='proj_concat', how='left')
df_dev['developer_clean'] = (
    df_dev['dev_raw']
    .fillna(df_dev['manual_developer'])
)
coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage after manual enrichment: {coverage:.1f}%")

Final developer coverage after manual enrichment: 54.0%


In [45]:
manual_map2 = pd.read_csv('developer_manual_mapping2.csv').drop(columns=['count'], errors='ignore')

df_dev = df_dev.merge(manual_map2, on='proj_concat', how='left')

df_dev['developer_clean'] = (
    df_dev['developer_clean']
    .fillna(df_dev['manual_developer2'])
)

coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage after manual enrichment: {coverage:.2f}%")

Final developer coverage after manual enrichment: 63.10%


# 4. Checking Unique Developers

In [48]:
df_dev['developer_clean'].nunique()

222

In [50]:
dev_names = sorted(df_dev['developer_clean'].dropna().unique())
pairs_contained = []

for i, a in enumerate(dev_names):
    for b in dev_names[i+1:]:
        # skip if identical
        if a == b: 
            continue
        # check containment (longer string includes shorter one)
        if a in b or b in a:
            pairs_contained.append((a, b, len(a)/len(b) if len(b)>0 else 0))

contain_df = pd.DataFrame(pairs_contained, columns=['dev1','dev2','length_ratio'])
display(contain_df.head(30))
print(f"Found {len(contain_df)} containment pairs.")

,dev1,dev2,length_ratio
0,citi developers,citi developers - tower 1,0.600000
1,citi developers,citi developers - tower 2,0.600000
2,condor,condor group,0.500000
3,ellington,ellington properties,0.450000
4,ellington,wellington,0.900000
5,fakhruddin,fakhruddin properties,0.476190
6,iman,iman developers,0.266667
7,irth,irth group,0.400000
8,london gate,london gate real estate development,0.314286
9,mag,mag,0.750000


Found 19 containment pairs.


In [52]:
dev_unique = sorted([d.strip().lower() for d in df_dev['developer_clean'].dropna().unique()])

# Create a dataframe of all pairs with similarity score
pairs = []
for a, b in combinations(dev_unique, 2):
    score = fuzz.token_sort_ratio(a, b)
    if score >= 85:  # tune threshold, 85–95 recommended
        pairs.append((a, b, score))

dev_pairs = pd.DataFrame(pairs, columns=['developer_1','developer_2','similarity']).sort_values('similarity', ascending=False)

print(f"Found {len(dev_pairs)} potentially duplicated developer pairs.")
display(dev_pairs.head(30))

Found 29 potentially duplicated developer pairs.


,developer_1,developer_2,similarity
21,mag,mag,100.000000
9,citi developers - tower 1,citi developers - tower 2,96.000000
11,ellington,wellington,94.736842
24,saas properties,saba properties,93.333333
28,tiger properties,time properties,90.322581
19,maaia developers,maas developers,90.322581
25,saba properties,sbk properties,89.655172
16,hre development,hz development,89.655172
13,h&h development,hz development,89.655172
2,asak real estate development,deca real estate development,89.285714


In [54]:
merge_map = {
    'citi developers - tower 1': 'citi developers',
    'citi developers - tower 2': 'citi developers',
    'condor group':'condor',
    'ellington properties':'ellington',
    'fakhruddin properties':'fakhruddin',
    'iman developers': 'iman',
    'irth group': 'irth',
    'london gate real estate development': 'london gate',
    'mag ':'mag',
    'majid al futtaim': 'majid',
    'mashriq elite development': 'mashriq elite',
    'nshama development': 'nshama',
    'prescott real estate development': 'prescott',
    'reportage properties': 'reportage',
    'tabeer developments': 'tabeer',
    'taraf developments': 'taraf',
    'union properties': 'union',
    'zimaya properties': 'zimaya',
    'al dar 1': 'aldar',
    'al dar 2': 'aldar',
    'athlon 1': 'aldar',
    'athlon 2': 'aldar',
    'athlon 3': 'aldar',
    'athlon 4': 'aldar',
    'haven 1': 'haven',
    'haven 2': 'haven' 
}

In [56]:
df_dev['developer_clean'] = df_dev['developer_clean'].replace(merge_map)
df_dev['developer_clean'].nunique()

197

# 5. Duplicated or Similar Projects

In [59]:
# Get known (already matched) and unknown projects
known_projects = (
    df_dev[df_dev['developer_clean'].notna()]
    [['project', 'developer_clean']]
    .drop_duplicates(subset='project')
)
unknown_projects = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['project'].notna() & (df_dev['project'].str.strip() != ""), ['project']]
    .drop_duplicates(subset='project')
)

# For each unknown project, find the most similar known project
matches = []
for proj in unknown_projects['project']:
    match = process.extractOne(
        proj,
        known_projects['project'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=88  # adjust threshold; 
    )
    if match:
        matched_proj, score, _ = match
        matched_dev = known_projects.loc[known_projects['project']==matched_proj, 'developer_clean'].iloc[0]
        matches.append((proj, matched_proj, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_project','matched_project','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched projects have strong similarity (>=88) with known projects.")
display(propagation_df.head())

131 unmatched projects have strong similarity (>=88) with known projects.


,unmatched_project,matched_project,developer_suggested,similarity
0,emirates living - springs 2,emirates living - springs 1,emaar,96.296296
1,emirates living - springs 12,emirates living - springs 1,emaar,98.181818
2,emirates living - springs 9,emirates living - springs 1,emaar,96.296296
3,emirates living - springs 3,emirates living - springs 1,emaar,96.296296
4,emirates living - springs 5,emirates living - springs 1,emaar,96.296296


In [61]:
# your exclusion list
excluded_unmatched = [
    'balqis residence', 'elle residences', 'laya residences',
    'olivia residences', 'liv residence', 'maya townhouses',
    'verdana 2', 'myka residence', 'aria', 'mr.c residences downtown',
    'elevia residences', 'azha downtown residences', 'riva residence',
    'bv residences', 'belmont residences', 'lua residences', 'nb residences'
]

# create propagation mapping excluding those
propagate_map = (
    propagation_df.loc[~propagation_df['unmatched_project'].isin(excluded_unmatched)]
    .set_index('unmatched_project')['developer_suggested']
    .to_dict()
)

# apply the mapping
df_dev['developer_clean'] = df_dev['developer_clean'].fillna(
    df_dev['project'].map(propagate_map)
)

# verify coverage after propagation
coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 66.83%


# 6. Duplicated or Similar Buildings

In [63]:
# Get known (already matched) and unknown buildings
known_buildings = (
    df_dev[df_dev['developer_clean'].notna()]
    [['building', 'developer_clean']]
    .drop_duplicates(subset='building')
)
unknown_buildings = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['building'].notna() & (df_dev['building'].str.strip() != ""), ['building']]
    .drop_duplicates(subset='building')
)

# For each unknown building, find the most similar known building
matches = []
for proj in unknown_buildings['building']:
    match = process.extractOne(
        proj,
        known_buildings['building'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90  # adjust threshold; 90 = strong match
    )
    if match:
        matched_proj, score, _ = match
        matched_dev = known_buildings.loc[known_buildings['building']==matched_proj, 'developer_clean'].iloc[0]
        matches.append((proj, matched_proj, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_building','matched_building','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched buildings have strong similarity (>=90) with known buildings.")
display(propagation_df.head())

27 unmatched buildings have strong similarity (>=90) with known buildings.


,unmatched_building,matched_building,developer_suggested,similarity
0,views 1,views 1,dubai south,100.000000
1,elite residences 3,elite residence,tameer holdings,90.909091
2,elite residence 1,elite residence,tameer holdings,93.750000
3,elite residences 2,elite residence,tameer holdings,90.909091
4,south residences,south residence 2,dubai south,90.909091


In [65]:
# your exclusion list
excluded_unmatched = [
    'jade residence', 'riviera residence', 'taya residences',
    'olivia residences', 'liv residence', 'dana tower', 'amalia residences'
]

# create propagation mapping excluding those
propagate_map = (
    propagation_df.loc[~propagation_df['unmatched_building'].isin(excluded_unmatched)]
    .set_index('unmatched_building')['developer_suggested']
    .to_dict()
)

# apply the mapping
df_dev['developer_clean'] = df_dev['developer_clean'].fillna(
    df_dev['building'].map(propagate_map)
)

# verify coverage after propagation
coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 67.15%


# 7. Splitting Project Name

In [69]:
# Function to split project names into prefix and suffix
def split_project_name(name):
    """
    Splits a project name into two parts:
    - prefix: everything before the first non-letter/non-space character
    - suffix: everything after that character
    If no such character exists, suffix is None.
    """
    if pd.isna(name):
        return pd.Series([None, None])
    
    match = re.split(r'[^A-Za-z\s]+', name, maxsplit=1)
    
    if len(match) == 1:
        return pd.Series([match[0].strip(), None])
    else:
        return pd.Series([match[0].strip(), match[1].strip()])

# Apply to your DataFrame
df_dev[['proj_prefix', 'proj_suffix']] = df_dev['project'].apply(split_project_name)

In [71]:
# Get known (already matched) and unknown projects
known_projects = (
    df_dev[df_dev['developer_clean'].notna()]
    [['proj_prefix', 'developer_clean']]
    .drop_duplicates(subset='proj_prefix')
)
unknown_projects = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['proj_prefix'].notna() & (df_dev['proj_prefix'].str.strip() != ""), ['proj_prefix']]
    .drop_duplicates(subset='proj_prefix')
)

# For each unknown project, find the most similar known project
matches = []
for proj in unknown_projects['proj_prefix']:
    match = process.extractOne(
        proj,
        known_projects['proj_prefix'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90  # adjust threshold; 90 = strong match
    )
    if match:
        matched_proj, score, _ = match
        matched_dev = known_projects.loc[known_projects['proj_prefix']==matched_proj, 'developer_clean'].iloc[0]
        matches.append((proj, matched_proj, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_proj_prefix','matched_proj_prefix','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched project-prefixes have strong similarity (>=90) with known project-prefixes.")
display(propagation_df.head())

34 unmatched project-prefixes have strong similarity (>=90) with known project-prefixes.


,unmatched_proj_prefix,matched_proj_prefix,developer_suggested,similarity
0,elite,elite,tameer holdings,100.000000
1,balqis residence,bali residences,serene developments,90.322581
2,avenue residence,avenue residence,nabni,100.000000
3,midtown,midtown,deyaar,100.000000
4,maple,maple,emaar,100.000000


In [73]:
# your exclusion list
excluded_prefixes = [
    'balqis residence', 'elle residences', 'axis residences', 'the', 'amalia residences',
    'laya residences', 'olivia residences', 'verdana', 'liv residence', 'the residence',
    'maya residences', 'maya townhouses', 'serra tower', 'prive residence',
    'f', 'c', 'may residence tower', 'jade residence', 'riviera residence',
    'g', 'e', 'taya residences', 'vida residences', 'dana tower', 'd', 'building'
]

# Filter matches excluding unwanted prefixes
valid_matches = propagation_df[
    ~propagation_df['unmatched_proj_prefix'].str.lower().isin(excluded_prefixes)
].copy()

# Apply mapping to df_dev
mapping_dict = dict(zip(valid_matches['unmatched_proj_prefix'], valid_matches['developer_suggested']))

df_dev['developer_clean'] = df_dev['developer_clean']  # keep existing values
df_dev.loc[
    df_dev['proj_prefix'].isin(mapping_dict.keys()) & df_dev['developer_clean'].isna(),
    'developer_clean'
] = df_dev['proj_prefix'].map(mapping_dict)

final_cov = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage: {final_cov:.1f}%")

Final developer coverage: 68.9%


# 8. Splitting Building Name

In [76]:
# Apply to your DataFrame
df_dev[['building_prefix', 'building_suffix']] = df_dev['building'].apply(split_project_name)

In [78]:
# Get known (already matched) and unknown projects
known_buildings = (
    df_dev[df_dev['developer_clean'].notna()]
    [['building_prefix', 'developer_clean']]
    .drop_duplicates(subset='building_prefix')
)
unknown_buildings = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['building_prefix'].notna() & (df_dev['building_prefix'].str.strip() != ""), ['building_prefix']]
    .drop_duplicates(subset='building_prefix')
)

# For each unknown project, find the most similar known project
matches = []
for build in unknown_buildings['building_prefix']:
    match = process.extractOne(
        build,
        known_buildings['building_prefix'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90  # adjust threshold; 90 = strong match
    )
    if match:
        matched_build, score, _ = match
        matched_dev = known_buildings.loc[known_buildings['building_prefix']==matched_build, 'developer_clean'].iloc[0]
        matches.append((build, matched_build, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_building_prefix','matched_building_prefix','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched project-prefixes have strong similarity (>=95) with known project-prefixes.")
display(propagation_df.head())

28 unmatched project-prefixes have strong similarity (>=95) with known project-prefixes.


,unmatched_building_prefix,matched_building_prefix,developer_suggested,similarity
0,saba,saba,saba properties,100.000000
1,the residences ii,the residences,al habtoor,90.322581
2,balqis residence,bali residences,serene developments,90.322581
3,elle residences,elite residences,tameer holdings,90.322581
4,serra tower,terra tower,dugasta properties,90.909091


In [80]:
# your exclusion list
excluded_prefixes = [
    'balqis residence', 'elle residences', 'axis residences', 'the', 'amalia residences',
    'laya residences', 'olivia residences', 'verdana', 'liv residence', 'the residence',
    'maya residences', 'maya townhouses', 'serra tower', 'prive residence',
    'f', 'c', 'may residence tower', 'jade residence', 'riviera residence',
    'g', 'e', 'taya residences', 'vida residences', 'dana tower', 'd', 'building'
]

# Filter matches excluding unwanted prefixes
valid_matches = propagation_df[
    ~propagation_df['unmatched_building_prefix'].str.lower().isin(excluded_prefixes)
].copy()

# Apply mapping to df_dev
mapping_dict = dict(zip(valid_matches['unmatched_building_prefix'], valid_matches['developer_suggested']))

df_dev['developer_clean'] = df_dev['developer_clean']  # keep existing values
df_dev.loc[
    df_dev['building_prefix'].isin(mapping_dict.keys()) & df_dev['developer_clean'].isna(),
    'developer_clean'
] = df_dev['building_prefix'].map(mapping_dict)

final_cov = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage: {final_cov:.1f}%")

Final developer coverage: 69.0%


# 9. Merging and Saving

In [83]:
# Adapting the original df to merge
df['proj_concat'] = (
    df['master_project'] + ' | ' +
    df['project'] + ' | ' +
    df['building']
).str.strip()

In [85]:
# Keeping only project and developer_clean
df_dev_clean = df_dev[['proj_concat', 'developer_clean']].drop_duplicates(subset='proj_concat')

In [87]:
# Merging developer info into main dataset
df_merged = df.merge(df_dev_clean, on='proj_concat', how='left')

# Checking coverage after merge
coverage = df_merged['developer_clean'].notna().mean()
print(f"Developer field coverage after merge: {coverage:.1%}")

Developer field coverage after merge: 69.0%


In [89]:
df_merged.head()

,transaction_type,procedure,date,property_type,registration,area,building,project,master_project,landmark,...,rooms,parking,size,price,meter_price,no_sellers,no_buyers,no_third_parties,proj_concat,developer_clean
0,Sales,Sell,2025-05-27,Villa,Existing Properties,Mirdif,,,,Dubai International Airport,...,6 B/R,0,1520.45,5500000.0,3617.35,1,1,0,| |,NaN
1,Mortgages,Mortgage Registration,2025-10-16,Villa,Existing Properties,Mirdif,,,,Dubai International Airport,...,5 B/R,0,696.77,2750000.0,3946.78,1,1,0,| |,NaN
2,Gifts,Grant,2025-07-03,Villa,Existing Properties,Mirdif,,,,Dubai International Airport,...,6 B/R,0,6967.73,42500001.0,6099.55,1,1,0,| |,NaN
3,Sales,Sell,2025-01-23,Villa,Existing Properties,Abu Hail,,,,Dubai International Airport,...,4 B/R,0,231.64,1350000.0,5828.01,1,1,0,| |,NaN
4,Sales,Sell - Pre registration,2025-05-14,Unit,Off-Plan Properties,Burj Khalifa,volta tower,volta tower,,Burj Khalifa,...,4 B/R,1,220.14,6659000.0,30248.93,1,1,0,| volta tower | volta tower,NaN


In [91]:
# Removing building and master_project columns
df_merged = df_merged.drop(columns=['building', 'master_project','proj_concat'], errors='ignore')

In [93]:
df_merged.head()

,transaction_type,procedure,date,property_type,registration,area,project,landmark,metro,mall,rooms,parking,size,price,meter_price,no_sellers,no_buyers,no_third_parties,developer_clean
0,Sales,Sell,2025-05-27,Villa,Existing Properties,Mirdif,,Dubai International Airport,Etisalat Metro Station,City Centre Mirdif,6 B/R,0,1520.45,5500000.0,3617.35,1,1,0,NaN
1,Mortgages,Mortgage Registration,2025-10-16,Villa,Existing Properties,Mirdif,,Dubai International Airport,Rashidiya Metro Station,City Centre Mirdif,5 B/R,0,696.77,2750000.0,3946.78,1,1,0,NaN
2,Gifts,Grant,2025-07-03,Villa,Existing Properties,Mirdif,,Dubai International Airport,Rashidiya Metro Station,City Centre Mirdif,6 B/R,0,6967.73,42500001.0,6099.55,1,1,0,NaN
3,Sales,Sell,2025-01-23,Villa,Existing Properties,Abu Hail,,Dubai International Airport,Abu Baker Al Siddique Metro Station,Missing,4 B/R,0,231.64,1350000.0,5828.01,1,1,0,NaN
4,Sales,Sell - Pre registration,2025-05-14,Unit,Off-Plan Properties,Burj Khalifa,volta tower,Burj Khalifa,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,4 B/R,1,220.14,6659000.0,30248.93,1,1,0,NaN


In [95]:
# Replacing missing project names with 'missing'
df_merged['project'] = df_merged['project'].replace('', pd.NA)  # make sure empty strings become NA first
df_merged['project'] = df_merged['project'].fillna('missing')
df_merged['project'].value_counts()

project
missing              22960
binghatti skyrise     2673
sobha solis           2066
binghatti elite       1690
skyvue                1620
                     ...  
fairway villas 3         1
j-haus residences        1
haven villas             1
arib collection          1
park villa's             1
Name: count, Length: 2379, dtype: int64

In [97]:
# Renaming developer column and replace missing values
df_merged = df_merged.rename(columns={'developer_clean': 'developer'})
df_merged['developer'] = df_merged['developer'].fillna('missing')
df_merged['developer'].value_counts()

developer
missing                   68380
binghatti                 19438
emaar                     17506
sobha                     11847
damac                     10892
                          ...  
bentley home                  6
nexus                         4
al marina                     4
newbury developments          1
casa vista development        1
Name: count, Length: 198, dtype: int64

In [106]:
# Saving the final dataset
#df_merged.to_csv("re_2025_analysis.csv", index=False)
#print("✅ Final dataset saved as 're_2025_analysis.csv'")

✅ Final dataset saved as 're_2025_analysis.csv'
